# Web development with Flask
* in lesson 14 we already learned the basics of HTTP (methods, status codes) and even built our own tiny JSON api using the standard library's `http.server`
* but writing your own `BaseHTTPRequestHandler` subclass by hand is NOT how real world python web apps get built - almost nobody does that in a real project
* instead, real projects use a **web framework** - a library that already solved routing, request parsing, response building etc. for you, so you just plug in your own logic
* one of the most popular, beginner-friendly python web frameworks is **Flask** - it calls itself a "micro-framework" because its core is tiny and it doesn't force many decisions on you, you add extensions/libraries as you need them
* two other big names you will hear a lot in the python web world:
  * **FastAPI** - a more modern framework, built around async code and automatic interactive api documentation, very popular for building JSON apis
  * **Django** - a much bigger "batteries-included" framework that ships with a built-in admin panel, a database ORM, user authentication and much more out of the box, very popular for building full websites
* this lesson focuses on **Flask** specifically because its simplicity makes it the easiest place to actually understand what a web framework does for you

## a note on how we run Flask in this notebook
* normally you would start a Flask app for real and it would listen on a real network port, either with `flask run` (the built-in development server, only meant for local development) or, in production, behind a proper WSGI server like `gunicorn` or `uwsgi` (the dev server is explicitly NOT safe/fast enough for real traffic)
* in a notebook, spinning up a real background server (like lesson 14 did with `http.server` + `threading`) gets fragile with Flask: newer versions of Flask/Werkzeug removed the old trick of shutting the dev server down from inside a request, so a real `app.run()` in a background thread can just hang or leave a zombie thread behind
* Flask actually ships its own solution for this: `app.test_client()` - it lets us call our routes directly, in-process, with zero real sockets and zero servers involved at all
* this is literally the officially recommended way to test a flask app, so we are not even cutting corners here - we use `test_client()` for every single interaction with our app in this whole lesson
* one more small side-effect of this: flask "locks" an app once it has handled its first request - you can no longer add new `@app.route(...)`s to that SAME app object afterwards (it raises an `AssertionError` if you try, on purpose - real apps are supposed to fully set themselves up before they ever start serving traffic)
* thats why below we create a fresh little `Flask(__name__)` app in most sections - one self-contained example at a time, all its routes registered before we ever call `test_client()` on it

## creating a minimal Flask app
* `pip install flask` gets us the framework (already done for this lesson)
* `Flask(__name__)` creates our application object - basically "the app" that everything else gets attached to
* the `@app.route("/some/path")` decorator (remember decorators from lesson 09!) registers a plain python function as the code that runs when someone requests that path
* the function just returns a value - Flask takes care of turning it into a proper HTTP response (status line, headers, body) for us, no manual `send_header`/`wfile.write()` needed like in lesson 14

In [1]:
from flask import Flask  # the core Flask class, everything starts here

# __name__ tells flask where "here" is (which module/package), it uses this internally to find things like a templates/ folder
app = Flask(__name__)


# the decorator says: "whenever someone requests '/', run this function to build the response"
@app.route("/")
def home():
    # a view function (thats the official flask name for a route-handling function) can just return a plain string
    # flask automatically wraps this into a 200 OK response with Content-Type text/html
    return "hello from our very first flask app!"


# app.test_client() gives us a fake client that talks to our app WITHOUT any real network/socket/thread involved
client = app.test_client()

# .get(path) simulates a real GET request to that path, entirely in-process
response = client.get("/")

print("status code:", response.status_code)          # should be 200, same status code concept as lesson 14
print("raw data (bytes):", response.data)              # the body always comes back as bytes first, just like a real http response
print("as text:", response.get_data(as_text=True))     # .get_data(as_text=True) decodes it to a normal python string for us

status code: 200
raw data (bytes): b'hello from our very first flask app!'
as text: hello from our very first flask app!


## routes with url parameters
* often we want a route to react to a value thats part of the url itself, not a separate query parameter
* `<name>` inside a route string captures that part of the url and passes it as an argument to our view function, with the same name
* this is a very common pattern for "get me one specific thing" routes, e.g. `/users/<user_id>` or here `/greet/<name>`

In [2]:
app = Flask(__name__)  # a fresh app for this example (our "home" app above already handled a request, so it cant take new routes anymore)


@app.route("/greet/<name>")
def greet(name):
    # "name" here is exactly the <name> part flask captured out of the url, flask passes it straight into our function
    return f"hey there, {name}! welcome to flask"


client = app.test_client()  # test_client() is cheap to create, we make a fresh one for each fresh app

# lets call it with a couple of different names, to see the url parameter actually changing the response
response_alex = client.get("/greet/Alex")
response_mia = client.get("/greet/Mia")

print("greeting for Alex:", response_alex.get_data(as_text=True))
print("greeting for Mia:", response_mia.get_data(as_text=True))

greeting for Alex: hey there, Alex! welcome to flask
greeting for Mia: hey there, Mia! welcome to flask


## returning HTML
* a view function can return any string, including actual HTML markup - the browser (or here, our test client) just receives raw text either way
* writing HTML directly as a python f-string works, but gets messy fast and is a bit risky (more on that in a second)
* Flask ships with the **Jinja2** templating engine, which lets us write a template string with `{{ placeholder }}` syntax that gets filled in with real values via `render_template_string(...)`
* in a real project you would NOT inline your HTML as a python string at all - you'd put templates as separate `.html` files inside a `templates/` folder next to your app, and load them with `render_template("mytemplate.html", ...)` instead of `render_template_string(...)` - we use the "string" version here purely so this lesson doesn't need to create any files on disk

In [3]:
from flask import render_template_string

app = Flask(__name__)  # fresh app again for this example, same reason as before


# option 1: just build the html ourselves with an f-string
@app.route("/hello_html/<name>")
def hello_html(name):
    # this WORKS, but if "name" ever contained something like "<script>...</script>" this would just inject it raw into the page (a classic security bug called XSS)
    return f"<h1>Hello {name}!</h1><p>this is plain html we built ourselves</p>"


# option 2: the more proper way, using jinja2 templating via render_template_string
@app.route("/hello_template/<name>")
def hello_template(name):
    # {{ name }} is a jinja2 placeholder - it gets replaced with the value we pass in as a keyword argument below
    # jinja2 also automatically "escapes" the value for us (turns dangerous characters into safe html entities), so this version is actually safer too
    return render_template_string("<h1>Hello {{ name }}!</h1><p>this came from a jinja2 template string</p>", name=name)


client = app.test_client()

response_html = client.get("/hello_html/Sam")
response_template = client.get("/hello_template/Sam")

print("plain f-string html:", response_html.get_data(as_text=True))
print("jinja2 template html:", response_template.get_data(as_text=True))

plain f-string html: <h1>Hello Sam!</h1><p>this is plain html we built ourselves</p>
jinja2 template html: <h1>Hello Sam!</h1><p>this came from a jinja2 template string</p>


## returning JSON
* lesson 14 built a whole JSON api by hand: `json.dumps(payload).encode("utf-8")`, manually setting the `Content-Type` header, manually setting `Content-Length`, ...
* Flask cuts basically all of that away for us
* `jsonify({...})` takes a python dict (or list) and returns a proper JSON response, with the correct `Content-Type: application/json` header already set
* even simpler: modern flask lets a view function just `return` a plain python dict directly, and it gets auto-converted to JSON for us - no `jsonify` call needed at all!

In [4]:
from flask import jsonify

app = Flask(__name__)  # fresh app again

# a small "database" of one fake user, just for this example - a plain python dict works fine
FAKE_USER = {"id": 1, "name": "Alex", "role": "developer"}


@app.route("/user_jsonify")
def user_jsonify():
    # jsonify() builds us a full response object: correct headers + json-encoded body, one function call, thats it
    return jsonify(FAKE_USER)


@app.route("/user_dict")
def user_dict():
    # modern flask (2.x+) is smart enough to auto-convert a returned dict into a json response for us, no jsonify needed
    return FAKE_USER


client = app.test_client()

response_a = client.get("/user_jsonify")
response_b = client.get("/user_dict")

print("jsonify() version - content type:", response_a.content_type)
print("jsonify() version - parsed json:", response_a.get_json())  # .get_json() is the test_client/response equivalent of requests' .json() from lesson 14

print("plain dict version - content type:", response_b.content_type)
print("plain dict version - parsed json:", response_b.get_json())

# compare this whole cell to the http.server version in lesson 14 - no manual send_header, no manual json.dumps().encode() anywhere

jsonify() version - content type: application/json
jsonify() version - parsed json: {'id': 1, 'name': 'Alex', 'role': 'developer'}
plain dict version - content type: application/json
plain dict version - parsed json: {'id': 1, 'name': 'Alex', 'role': 'developer'}


## handling different HTTP methods on the same route
* one url can react differently depending on the HTTP method used, exactly the REST idea from lesson 14 (`GET /items` = read, `POST /items` = create, ...)
* `methods=["GET", "POST"]` on the `@app.route()` decorator says "this route accepts both of these methods"
* inside the view function we check `request.method` to know which one we're currently handling
* `request` is a special flask object that always describes the CURRENT incoming request - `request.get_json()` reads a json body from a POST, `request.args.get(...)` reads a query parameter from the url (the `?key=value` part, same concept as lesson 14's `query_received`)

In [5]:
from flask import request

app = Flask(__name__)  # fresh app again

# a plain python list acts as our tiny "storage" for this example, just so GET has something to show after a POST
LOGGED_MESSAGES = []


@app.route("/messages", methods=["GET", "POST"])
def messages():
    if request.method == "POST":
        # request.get_json() parses the request body as json and gives us back a python dict, similar to response.json() on the client side
        body = request.get_json()
        LOGGED_MESSAGES.append(body.get("text", ""))
        return jsonify({"status": "stored", "total_messages": len(LOGGED_MESSAGES)}), 201  # returning (body, status_code) sets a custom status code
    else:
        # request.method == "GET" here - request.args behaves like a dict for query params, .get() lets us provide a default if its missing
        limit = int(request.args.get("limit", 10))  # query params always arrive as strings, we convert to int ourselves
        return jsonify({"messages": LOGGED_MESSAGES[:limit]})


client = app.test_client()

# .post(path, json=...) mirrors requests.post(url, json=...) from lesson 14 - it serializes our dict and sets the content type for us
post_response = client.post("/messages", json={"text": "hello flask"})
print("POST status:", post_response.status_code, "body:", post_response.get_json())

client.post("/messages", json={"text": "second message"})

# query_string=... lets test_client attach query parameters, same idea as requests' params= from lesson 14
get_response = client.get("/messages", query_string={"limit": 1})
print("GET status:", get_response.status_code, "body:", get_response.get_json())

POST status: 201 body: {'status': 'stored', 'total_messages': 1}
GET status: 200 body: {'messages': ['hello flask']}


## a small coherent mini-app: a todo list api
* lets tie several routes together into one tiny but realistic api, the classic "todo list" example
* `GET /todos` - list every todo we currently have
* `POST /todos` - add a new todo, validating the input (recap of lesson 06's error handling ideas: check for bad input BEFORE it causes problems, and return a clear error instead of crashing or silently doing the wrong thing)
* `DELETE /todos/<todo_id>` - remove one specific todo by its id
* we store everything in a plain python list at module level, purely to keep this example simple and dependency-free - a REAL app would use an actual database instead (e.g. `sqlite3` from lesson 13), because a plain in-memory python list resets itself back to empty every single time the app restarts, which is obviously useless for real users

In [6]:
app = Flask(__name__)  # fresh app again - this becomes our little todo-app for the rest of this section

# our "database": just a list of dicts living in memory - gone the moment the process restarts (unlike a real sqlite3 database file from lesson 13)
TODOS = []
NEXT_ID = 1  # a simple counter we use to hand out unique ids, a real database would normally do this for us automatically


@app.route("/todos", methods=["GET", "POST"])
def todos_collection():
    global NEXT_ID  # we need "global" here because we reassign NEXT_ID inside the function (see lesson 04 for more on scope)

    if request.method == "GET":
        return jsonify({"todos": TODOS})

    # from here on we're handling POST /todos (adding a new todo)
    body = request.get_json(silent=True)  # silent=True makes it return None instead of raising an error if the body isnt valid json at all

    # input validation, same spirit as lesson 06's error handling - check for problems explicitly and react on purpose, dont just assume the caller sent good data
    if not body or "title" not in body or not str(body["title"]).strip():
        # 400 Bad Request is exactly the status code lesson 14 introduced for "the client sent something wrong"
        return jsonify({"error": "a non-empty 'title' field is required"}), 400

    new_todo = {"id": NEXT_ID, "title": body["title"], "done": False}
    TODOS.append(new_todo)
    NEXT_ID += 1  # always increase the counter, so the next todo gets a fresh, still-unique id

    return jsonify(new_todo), 201  # 201 Created, same convention as lesson 14


@app.route("/todos/<int:todo_id>", methods=["DELETE"])
def todos_item(todo_id):
    # "<int:todo_id>" is a "converter" - flask automatically parses this url segment as an int for us, instead of us having to int() it ourselves
    global TODOS
    matching = [todo for todo in TODOS if todo["id"] == todo_id]  # list comprehension from lesson 03, filtering for the one todo (if any) with this id

    if not matching:
        return jsonify({"error": f"no todo with id {todo_id}"}), 404  # 404 Not Found, same as lesson 14's "no such route" case but for a missing RESOURCE this time

    TODOS = [todo for todo in TODOS if todo["id"] != todo_id]  # rebuild the list without the deleted todo
    return jsonify({"status": "deleted", "id": todo_id})


client = app.test_client()

# add two todos
r1 = client.post("/todos", json={"title": "learn flask"})
r2 = client.post("/todos", json={"title": "build a real project"})
print("created todo 1:", r1.status_code, r1.get_json())
print("created todo 2:", r2.status_code, r2.get_json())

# try adding an invalid one (missing title) - should get our 400 error, not a crash
r_bad = client.post("/todos", json={"not_a_title": "oops"})
print("bad request:", r_bad.status_code, r_bad.get_json())

# list them all
print("all todos:", client.get("/todos").get_json())

# delete the first one
r_delete = client.delete("/todos/1")
print("delete result:", r_delete.status_code, r_delete.get_json())

# deleting an id that doesnt exist (anymore) - should be a 404
r_delete_missing = client.delete("/todos/999")
print("delete missing:", r_delete_missing.status_code, r_delete_missing.get_json())

print("todos after deletion:", client.get("/todos").get_json())

created todo 1: 201 {'done': False, 'id': 1, 'title': 'learn flask'}
created todo 2: 201 {'done': False, 'id': 2, 'title': 'build a real project'}
bad request: 400 {'error': "a non-empty 'title' field is required"}
all todos: {'todos': [{'done': False, 'id': 1, 'title': 'learn flask'}, {'done': False, 'id': 2, 'title': 'build a real project'}]}
delete result: 200 {'id': 1, 'status': 'deleted'}
delete missing: 404 {'error': 'no todo with id 999'}
todos after deletion: {'todos': [{'done': False, 'id': 2, 'title': 'build a real project'}]}


## basic error handling
* just like our own `http.server` handler in lesson 14 had an `else` branch for unknown routes, flask has its own built-in fallback for a route that doesn't exist at all: a `404 Not Found`
* by default this comes back as an HTML error page (fine for a browser, not great for a JSON api) - `@app.errorhandler(404)` lets us override that with our own response, e.g. a JSON error shape that matches the rest of our api

In [7]:
app = Flask(__name__)  # fresh app again - our previous todo-app already served requests, so it cant register anything new anymore


@app.errorhandler(404)
def handle_not_found(error):
    # this runs for ANY unmatched route on this app - our own custom 404 shape instead of flask's default html error page
    return jsonify({"error": "resource not found", "detail": str(error)}), 404


client = app.test_client()

# requesting a route that was never defined anywhere on our app
response = client.get("/this/route/does/not/exist")
print("status code:", response.status_code)          # still 404, flask's normal behaviour
print("body is now json:", response.get_json())        # but now its OUR json shape, not flasks default html page
print("content type:", response.content_type)          # application/json, confirming our errorhandler really took over

status code: 404
body is now json: {'detail': '404 Not Found: The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.', 'error': 'resource not found'}
content type: application/json


## quick comparison to other languages
* Flask is most directly comparable to **Node.js's Express** (`app.get("/route", (req, res) => { ... })`) or Ruby's **Sinatra** - all three are deliberately minimalist "micro-frameworks" where you add exactly what you need, and the routing-decorator-vs-routing-function feel is very similar
* Django (still python!) is a much closer match to Ruby on Rails, Java/Kotlin's Spring Boot, or C#'s ASP.NET Core - all "batteries-included" frameworks that bundle routing + an ORM + templating + authentication together out of the box
* Go's standard `net/http` package is deliberately closer to Flask's minimalism than to Django's completeness - Go's whole ecosystem tends to favor small, composable pieces over one giant framework, similar in spirit to python's Flask/FastAPI camp
* but nearly every single one of these frameworks converges on the exact same core idea this lesson demonstrated: map an HTTP method+path to a handler function, read the request (params/body/headers), return a response (html/json/status code) - the syntax differs everywhere, the concept is identical everywhere

## Exercises
1. add a new flask route `/square/<int:number>` that returns the square of the given number as json (e.g. requesting `/square/5` should return something like `{"number": 5, "square": 25}`)
2. extend the todo app with a new route that marks a todo as done by its id (e.g. `PUT /todos/<int:todo_id>/done` that flips that todo's `"done"` field to `True`)
3. add a `@app.errorhandler(404)` to a *fresh* flask app that returns a custom json error message instead of flask's default html 404 page, and verify it with `test_client()`
4. add basic validation to a `POST /todos`-style route so it returns a 400 status code with an error message if the request is missing a `"title"` field (you can reuse the idea from the mini-app above, but write it again here yourself)

In [8]:
# TODO: solve exercise 1 here (add a /square/<int:number> route and test it with test_client())


# TODO: solve exercise 2 here (add a route that marks a todo as done by id)


# TODO: solve exercise 3 here (a fresh flask app with a custom @app.errorhandler(404))


# TODO: solve exercise 4 here (validate that POST /todos requires a "title" field, return 400 if missing)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [9]:
# sample solution 1 - /square/<int:number>
square_app = Flask(__name__ + "_exercise1")  # fresh app, all our earlier apps already served a request and cant take new routes anymore

@square_app.route("/square/<int:number>")
def square_route(number):
    return jsonify({"number": number, "square": number ** 2})

square_client = square_app.test_client()
print("square of 5:", square_client.get("/square/5").get_json())
print("square of 12:", square_client.get("/square/12").get_json())


# sample solution 2 - mark a todo as done by id
# we build a fresh little app that reuses the SAME idea/shape as the todo mini-app above (GET/POST /todos + a new PUT .../done route)
todo_app_v2 = Flask(__name__ + "_exercise2")
TODOS_V2 = []       # a fresh in-memory list for this fresh app, same concept as TODOS earlier
NEXT_ID_V2 = 1

@todo_app_v2.route("/todos", methods=["GET", "POST"])
def todos_v2_collection():
    global NEXT_ID_V2
    if request.method == "GET":
        return jsonify({"todos": TODOS_V2})
    body = request.get_json(silent=True)
    if not body or "title" not in body or not str(body["title"]).strip():
        return jsonify({"error": "a non-empty 'title' field is required"}), 400
    new_todo = {"id": NEXT_ID_V2, "title": body["title"], "done": False}
    TODOS_V2.append(new_todo)
    NEXT_ID_V2 += 1
    return jsonify(new_todo), 201

@todo_app_v2.route("/todos/<int:todo_id>/done", methods=["PUT"])
def mark_todo_done(todo_id):
    # this is the actual new part exercise 2 asked for - find the todo and flip its "done" field to True
    matching = [todo for todo in TODOS_V2 if todo["id"] == todo_id]
    if not matching:
        return jsonify({"error": f"no todo with id {todo_id}"}), 404
    matching[0]["done"] = True  # dicts are mutable, so this updates the todo directly inside the TODOS_V2 list too
    return jsonify(matching[0])

todo_client_v2 = todo_app_v2.test_client()  # all routes are registered above BEFORE we make any request, so flask stays happy
demo_todo = todo_client_v2.post("/todos", json={"title": "practice more flask"}).get_json()
print("fresh demo todo:", demo_todo)

done_response = todo_client_v2.put(f"/todos/{demo_todo['id']}/done")
print("after marking done:", done_response.status_code, done_response.get_json())


# sample solution 3 - a FRESH app with its own custom 404 errorhandler
exercise_app = Flask(__name__ + "_exercise3")  # a separate app object, so it doesnt clash with routes we already registered above

@exercise_app.errorhandler(404)
def exercise_404(error):
    return jsonify({"error": "nothing here, sorry!"}), 404

exercise_client = exercise_app.test_client()
missing_response = exercise_client.get("/nowhere")
print("custom 404 status:", missing_response.status_code)
print("custom 404 body:", missing_response.get_json())


# sample solution 4 - validating POST /todos requires a "title" field (written again from scratch here, own tiny app)
solution_app = Flask(__name__ + "_exercise4")
solution_todos = []

@solution_app.route("/todos", methods=["POST"])
def solution_create_todo():
    body = request.get_json(silent=True)
    if not body or "title" not in body or not str(body["title"]).strip():
        return jsonify({"error": "a non-empty 'title' field is required"}), 400
    solution_todos.append({"title": body["title"]})
    return jsonify({"status": "created", "title": body["title"]}), 201

solution_client = solution_app.test_client()
good = solution_client.post("/todos", json={"title": "valid todo"})
bad = solution_client.post("/todos", json={"description": "missing the title field"})
print("valid post:", good.status_code, good.get_json())
print("invalid post:", bad.status_code, bad.get_json())

square of 5: {'number': 5, 'square': 25}
square of 12: {'number': 12, 'square': 144}
fresh demo todo: {'done': False, 'id': 1, 'title': 'practice more flask'}
after marking done: 200 {'done': True, 'id': 1, 'title': 'practice more flask'}
custom 404 status: 404
custom 404 body: {'error': 'nothing here, sorry!'}
valid post: 201 {'status': 'created', 'title': 'valid todo'}
invalid post: 400 {'error': "a non-empty 'title' field is required"}


* congratulation you finished this lesson

## what we learned
* why real world python web apps almost always use a framework (Flask/FastAPI/Django) instead of a hand-rolled `http.server` handler like in lesson 14, and how Flask/FastAPI/Django roughly differ from each other
* creating a minimal Flask app with `Flask(__name__)` and the `@app.route()` decorator
* why we use `app.test_client()` for every interaction in this notebook (no real sockets/threads/servers, the officially recommended way to test flask apps) instead of running a real dev server in a background thread
* capturing url parameters directly in a route (`/greet/<name>`, `/todos/<int:todo_id>`)
* returning HTML directly vs. through jinja2's `render_template_string()` (and that real projects use a separate `templates/` folder with `render_template()` instead)
* returning JSON with `jsonify()` or just returning a plain dict, and how much boilerplate that saves compared to lesson 14's manual `http.server` approach
* handling multiple HTTP methods on one route via `request.method`, reading a json body with `request.get_json()` and query params with `request.args.get()`
* building a small coherent todo-list mini api (`GET`/`POST /todos`, `DELETE /todos/<id>`) backed by a simple in-memory list, including basic input validation that returns a `400` on bad input (and why a real app would use a real database like `sqlite3` from lesson 13 instead of an in-memory list)
* flask's built-in 404 behaviour and overriding it with a custom `@app.errorhandler(404)`
* how Flask compares to Express/Sinatra (minimalist camp) vs. Django/Rails/Spring Boot/ASP.NET Core (batteries-included camp), and that basically every web framework in every language boils down to the same idea: method+path -> handler -> response